# 02. Randomized Experiments

## Background

Randomized controlled trials are the epistemological gold standard for causal inference — not because they're the most common design, but because they make the identification assumptions transparent and verifiable. When treatment is assigned by a coin flip, the treated and control groups are, in expectation, identical on every confounder — observed and unobserved. The backdoor paths are cut at the source.

But "randomized" is not a binary. Practical RCTs face non-compliance, attrition, contamination, and small samples. Understanding exactly *why* randomization works — and where it can fail — is prerequisite knowledge for every observational method that follows. Each observational estimator is essentially an attempt to recover the properties of a randomized experiment from data that wasn't generated by one.

## What You'll Learn

- Why randomization achieves balance in expectation: E[Y(0)|T=1] = E[Y(0)|T=0]
- Covariate balance checking: standardized mean differences, love plots
- Fisher's exact randomization test — a permutation-based test with no parametric assumptions
- Neyman's repeated-sampling variance and when robust SEs matter
- Common threats: non-compliance (ITT vs TOT), attrition, contamination

## Why This Matters

Product experiments (A/B tests), clinical trials, and policy evaluations all start here. Balance checking should be mandatory before any analysis — an "out-of-spec" randomization that produced a large imbalance on an important covariate should be diagnosed before looking at outcomes. Fisher's randomization test is the right tool when you have small samples, clustered assignment, or non-normal outcomes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Why Randomization Works

With random assignment P(T=1) independent of {Y(0), Y(1), X}:

E[Y_obs | T=1] = E[Y(1) | T=1] = E[Y(1)]   (exchangeability)
E[Y_obs | T=0] = E[Y(0) | T=0] = E[Y(0)]

So the simple difference-in-means is an unbiased estimator of ATE.

In [ ]:
def simulate_rct(n=500, ate=2.0, seed=0):
    """Simulate a clean RCT with a single confounder."""
    rng = np.random.default_rng(seed)
    # Covariate correlated with outcome but NOT with treatment (RCT)
    age = rng.normal(40, 10, n)
    # Potential outcomes depend on age
    y0 = 0.3 * age + rng.normal(0, 2, n)
    y1 = y0 + ate + rng.normal(0, 0.5, n)
    # Random treatment assignment
    treated = rng.binomial(1, 0.5, n).astype(bool)
    y_obs = np.where(treated, y1, y0)
    return pd.DataFrame({'age': age, 'treated': treated, 'y_obs': y_obs,
                         'y0': y0, 'y1': y1})


df = simulate_rct(n=1000, ate=2.0)

# Difference-in-means
dim = df.loc[df.treated, 'y_obs'].mean() - df.loc[~df.treated, 'y_obs'].mean()
print(f"True ATE   = 2.000")
print(f"Naïve DiM  = {dim:.3f}  (unbiased because randomized)")

# Compare with confounded assignment
def simulate_observational(n=500, ate=2.0, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.normal(40, 10, n)
    y0 = 0.3 * age + rng.normal(0, 2, n)
    y1 = y0 + ate + rng.normal(0, 0.5, n)
    # Older people more likely treated (confounded!)
    p_treat = 1 / (1 + np.exp(-(0.1 * (age - 40))))
    treated = rng.binomial(1, p_treat).astype(bool)
    y_obs = np.where(treated, y1, y0)
    return pd.DataFrame({'age': age, 'treated': treated, 'y_obs': y_obs})

df_obs = simulate_observational(n=1000, ate=2.0)
dim_obs = df_obs.loc[df_obs.treated, 'y_obs'].mean() - df_obs.loc[~df_obs.treated, 'y_obs'].mean()
print(f"\nObservational DiM = {dim_obs:.3f}  (biased — older treated → higher baseline)") 

## 2. Covariate Balance Checking

Balance on pre-treatment covariates validates the randomization. The standard metric is the **Standardized Mean Difference (SMD)**:

SMD = (μ_treated − μ_control) / pooled_sd

Convention: |SMD| < 0.1 is well-balanced. A love plot shows all covariates at once.

In [ ]:
def smd(x_treated, x_control):
    """Standardized Mean Difference: (mu_t - mu_c) / pooled_sd"""
    mu_t, mu_c = np.mean(x_treated), np.mean(x_control)
    sd_t, sd_c = np.std(x_treated, ddof=1), np.std(x_control, ddof=1)
    pooled_sd = np.sqrt((sd_t**2 + sd_c**2) / 2)
    return (mu_t - mu_c) / pooled_sd


def love_plot(df, covariates, treatment_col='treated', ax=None):
    """Love plot: SMD for each covariate."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 0.4 * len(covariates) + 1.5))
    smds = {}
    for col in covariates:
        s = smd(df.loc[df[treatment_col], col], df.loc[~df[treatment_col], col])
        smds[col] = s

    cols_sorted = sorted(smds, key=lambda c: abs(smds[c]))
    smd_vals = [smds[c] for c in cols_sorted]
    colors = ['#F44336' if abs(v) > 0.1 else '#4CAF50' for v in smd_vals]

    ax.barh(cols_sorted, smd_vals, color=colors, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', lw=1)
    ax.axvline(0.1, color='orange', lw=1.5, ls='--', label='|SMD|=0.1 threshold')
    ax.axvline(-0.1, color='orange', lw=1.5, ls='--')
    ax.set_xlabel('Standardized Mean Difference')
    ax.set_title('Love Plot — Covariate Balance')
    ax.legend()
    return ax


# Simulate study with multiple covariates
np.random.seed(10)
n = 400
age   = np.random.normal(45, 12, n)
bmi   = np.random.normal(27, 5, n)
sbp   = np.random.normal(130, 15, n)
smoke = np.random.binomial(1, 0.25, n)
female = np.random.binomial(1, 0.5, n)

# RCT: pure random
treat_rct = np.random.binomial(1, 0.5, n).astype(bool)
# Observational: confounded by age and bmi
logit = -2 + 0.05 * age + 0.1 * bmi
treat_obs = np.random.binomial(1, 1/(1+np.exp(-logit)), n).astype(bool)

df_rct = pd.DataFrame({'treated':treat_rct,'age':age,'bmi':bmi,'sbp':sbp,'smoke':smoke,'female':female})
df_obs = pd.DataFrame({'treated':treat_obs,'age':age,'bmi':bmi,'sbp':sbp,'smoke':smoke,'female':female})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cov_cols = ['age','bmi','sbp','smoke','female']
love_plot(df_rct, cov_cols, ax=axes[0]); axes[0].set_title('RCT Balance (should all be near 0)')
love_plot(df_obs, cov_cols, ax=axes[1]); axes[1].set_title('Observational Imbalance (age, bmi confounded)')
plt.tight_layout()
plt.savefig('02_love_plot.png', dpi=110, bbox_inches='tight')
plt.show()

print("\nSMDs for observational data:")
for c in cov_cols:
    s = smd(df_obs.loc[df_obs.treated, c], df_obs.loc[~df_obs.treated, c])
    flag = ' ← imbalanced!' if abs(s) > 0.1 else ''
    print(f"  {c:8s}: {s:+.3f}{flag}")

## 3. Fisher's Randomization Test

Fisher (1935) proposed a test with *no distributional assumptions*: under the sharp null (H₀: Y_i(1) = Y_i(0) for all i — no effect for anyone), the observed data would look the same under any possible treatment assignment consistent with the design.

We compute the test statistic under each permutation of the assignment vector and compare to the observed value. The p-value is the fraction of permutations that produce a statistic as extreme as observed.

In [ ]:
def fisher_randomization_test(y, treated, statistic=np.mean, n_perms=5000, seed=0):
    """
    Fisher's exact randomization test.
    H0: sharp null (no individual treatment effects).
    statistic: function mapping (y[treated], y[control]) -> scalar.
    """
    rng = np.random.default_rng(seed)
    n_treated = treated.sum()
    n = len(y)

    # Observed test statistic
    t_obs = statistic(y[treated]) - statistic(y[~treated])

    # Permutation distribution
    null_dist = np.empty(n_perms)
    idx = np.arange(n)
    for i in range(n_perms):
        perm = rng.permutation(idx)
        t_mask = np.zeros(n, dtype=bool)
        t_mask[perm[:n_treated]] = True
        null_dist[i] = statistic(y[t_mask]) - statistic(y[~t_mask])

    p_value = np.mean(np.abs(null_dist) >= np.abs(t_obs))
    return t_obs, null_dist, p_value


# Test 1: real effect
np.random.seed(42)
n = 200
y_rct = np.random.normal(5, 2, n)
treated_rct = np.zeros(n, bool)
treated_rct[:100] = True
y_rct[treated_rct] += 1.5   # true effect

t_obs, null_dist, p_val = fisher_randomization_test(y_rct, treated_rct, n_perms=3000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (y_plot, t_plot, treated_plot, title) in zip(axes, [
    (y_rct, t_obs, treated_rct, f'True effect = 1.5\nFisher p = {p_val:.4f}'),
    (np.random.normal(5, 2, n), None, treated_rct, None),
]):
    if t_plot is None:
        y_null = y_plot.copy()
        t2, nd2, pv2 = fisher_randomization_test(y_null, treated_rct, n_perms=3000)
        t_plot, null_dist2, p_v2 = t2, nd2, pv2
        title = f'Null (no effect)\nFisher p = {pv2:.4f}'
        nd = nd2
    else:
        nd = null_dist

    ax.hist(nd, bins=60, density=True, alpha=0.7, color='steelblue', label='Null distribution')
    ax.axvline(t_plot, color='red', lw=2.5, label=f'Observed T={t_plot:.3f}')
    ax.set_xlabel('Test statistic (DiM)')
    ax.set_title(title)
    ax.legend()

plt.suptitle("Fisher's Randomization Test", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_fisher_test.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Extend to non-mean statistics — Fisher works for any test statistic
np.random.seed(7)
n = 100
# Simulate skewed outcomes (count data)
y_count = np.random.poisson(3, n)
treated_c = np.random.binomial(1, 0.5, n).astype(bool)
y_count[treated_c] += np.random.poisson(1, treated_c.sum())

# Compare mean-based vs median-based Fisher test
t_mean, nd_mean, p_mean = fisher_randomization_test(y_count, treated_c, statistic=np.mean)
t_med,  nd_med,  p_med  = fisher_randomization_test(y_count, treated_c, statistic=np.median)

print("Fisher test on count data:")
print(f"  Mean-based:   T={t_mean:.3f}, p={p_mean:.3f}")
print(f"  Median-based: T={t_med:.3f},  p={p_med:.3f}")
print("(Both valid — no distributional assumption needed)")

## 4. Neyman's Sampling-Based Variance

While Fisher's test conditions on *this specific sample*, Neyman (1923) asked about repeated sampling. The Neyman variance for DiM is conservative (ignores within-arm correlations between Y(0) and Y(1)), but is the standard in practice.

In [ ]:
def neyman_ate_inference(y_obs, treated, alpha=0.05):
    """ATE estimate with Neyman variance."""
    n = len(y_obs)
    n_t = treated.sum()
    n_c = n - n_t

    mu_t = y_obs[treated].mean()
    mu_c = y_obs[~treated].mean()
    ate_hat = mu_t - mu_c

    # Neyman variance (conservative)
    var_t = y_obs[treated].var(ddof=1) / n_t
    var_c = y_obs[~treated].var(ddof=1) / n_c
    se = np.sqrt(var_t + var_c)

    z = stats.norm.ppf(1 - alpha/2)
    ci_lo, ci_hi = ate_hat - z*se, ate_hat + z*se
    p_val = 2 * (1 - stats.norm.cdf(abs(ate_hat) / se))

    return dict(ate=ate_hat, se=se, ci=(ci_lo, ci_hi), p=p_val)


result = neyman_ate_inference(y_rct, treated_rct)
print(f"ATE estimate:  {result['ate']:.3f}")
print(f"SE:            {result['se']:.3f}")
print(f"95% CI:        [{result['ci'][0]:.3f}, {result['ci'][1]:.3f}]")
print(f"p-value:       {result['p']:.4f}")

## 5. Threats to Validity

| Threat | What happens | Fix |
|--------|-------------|-----|
| Non-compliance | Some assigned-to-treatment don't take it | Report ITT; use IV for LATE |
| Attrition / dropout | Outcome missing for some | Check if dropout correlates with treatment |
| Contamination | Control units receive treatment indirectly | Cluster randomization |
| SUTVA violation | Spillovers across units | Network/market-level analysis |
| Multiple testing | Many outcomes tested | Pre-registration + FDR correction |

In [ ]:
# Non-compliance: ITT vs TOT (Treatment-on-Treated)
np.random.seed(20)
n = 600
assigned = np.random.binomial(1, 0.5, n).astype(bool)
# 30% non-compliance: some assigned-to-treat don't take it
actually_treated = assigned.copy()
non_compliers = assigned & (np.random.binomial(1, 0.3, n).astype(bool))
actually_treated[non_compliers] = False

y0 = np.random.normal(5, 2, n)
y1 = y0 + 3.0  # true effect = 3 for compliers
y_obs = np.where(actually_treated, y1, y0)

# ITT: effect of assignment (intent-to-treat)
itt = y_obs[assigned].mean() - y_obs[~assigned].mean()
# Compliance rate
compliance = (actually_treated[assigned].sum() / assigned.sum())
# LATE (Local ATE = Complier ATE) via ratio
late = itt / compliance

print(f"Compliance rate:  {compliance:.1%}")
print(f"True effect:      3.000 (among compliers)")
print(f"ITT estimate:     {itt:.3f}  (diluted by non-compliers)")
print(f"LATE estimate:    {late:.3f}  (ITT / compliance rate ← Wald estimator)")
print(f"\nKey insight: ITT is conservative but always valid.")
print(f"LATE = TOT requires monotonicity assumption (no defiers).")